# 📈 Case Study: Stock Price Prediction using RNN
**Unit 3 – Sequence Modelling | CSEN3083 Deep Learning**

---

## Objective
Use a Recurrent Neural Network (RNN) to learn patterns from historical stock closing prices and predict the next day's price.

## What You Will Learn
- How RNNs model sequential / time-series data
- Sliding window approach for sequence creation
- Training and evaluating a regression RNN
- Visualising predictions vs actual prices

## Dataset
Live data pulled from **Yahoo Finance** using `yfinance` — no manual download needed.

---

## Step 1 — Install & Import Libraries

In [ ]:
# Install yfinance (not pre-installed on Colab)
!pip install yfinance --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries loaded successfully!")

## Step 2 — Download Stock Data
We use **Apple Inc. (AAPL)** as our stock. You can change the ticker to any valid Yahoo Finance symbol (e.g., `GOOGL`, `TSLA`, `RELIANCE.NS`).

In [ ]:
# ── Configuration ──────────────────────────────────────────
TICKER      = 'AAPL'        # Stock symbol
START_DATE  = '2018-01-01'  # Training data start
END_DATE    = '2024-01-01'  # Data end
WINDOW_SIZE = 60            # Look-back window (days)
# ───────────────────────────────────────────────────────────

# Download data directly from Yahoo Finance
print(f"Downloading {TICKER} stock data from {START_DATE} to {END_DATE}...")
df = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

# Keep only the closing price
df = df[['Close']].dropna()

print(f"\nTotal trading days: {len(df)}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
print("\nFirst 5 rows:")
df.head()

## Step 3 — Visualise Raw Stock Data

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df.index, df['Close'], color='steelblue', linewidth=1.2)
plt.title(f'{TICKER} Closing Price (2018–2024)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing
RNNs work best with normalised data. We scale prices to the range **[0, 1]** using MinMaxScaler.

**Sliding Window:** We create input-output pairs where:
- **X** = last 60 days of closing prices (sequence)
- **y** = the next day's closing price (target)

In [ ]:
# ── Normalise ───────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['Close']].values)

# ── Train / Test split (80 / 20) ────────────────────────────
train_size = int(len(scaled_data) * 0.80)
train_data = scaled_data[:train_size]
test_data  = scaled_data[train_size - WINDOW_SIZE:]  # overlap for windowing

print(f"Training samples : {train_size}")
print(f"Testing  samples : {len(scaled_data) - train_size}")

# ── Create sliding window sequences ─────────────────────────
def create_sequences(data, window):
    """Returns X (sequences) and y (next-step targets)."""
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, WINDOW_SIZE)
X_test,  y_test  = create_sequences(test_data,  WINDOW_SIZE)

# Reshape to (samples, time_steps, features) — required by Keras RNN
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"\nX_train shape: {X_train.shape}  →  (samples, time_steps, features)")
print(f"X_test  shape: {X_test.shape}")

## Step 5 — Build the RNN Model
A stacked SimpleRNN architecture:
- **Layer 1:** 64 RNN units — returns full sequences for the next layer
- **Dropout:** prevents overfitting
- **Layer 2:** 32 RNN units — returns final hidden state
- **Output:** 1 neuron (predicted next-day price)

In [ ]:
model = Sequential([
    # RNN layer 1 — return_sequences=True feeds output to next RNN layer
    SimpleRNN(units=64, activation='tanh', return_sequences=True,
              input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),

    # RNN layer 2 — return_sequences=False gives a single output vector
    SimpleRNN(units=32, activation='tanh', return_sequences=False),
    Dropout(0.2),

    # Output: one neuron for regression (price prediction)
    Dense(units=1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.summary()

## Step 6 — Train the Model

In [ ]:
# EarlyStopping stops training when validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## Step 7 — Plot Training & Validation Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Training Loss',   color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Model Loss During Training', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 — Make Predictions & Inverse Transform
The model predicts scaled values. We convert them back to actual USD prices.

In [ ]:
# Predict
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse-transform back to actual USD prices
y_pred  = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

print(f"Predicted shape : {y_pred.shape}")
print(f"Actual    shape : {y_actual.shape}")

## Step 9 — Evaluate the Model

In [ ]:
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mae  = mean_absolute_error(y_actual, y_pred)

# Mean Absolute Percentage Error
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100

print("=" * 40)
print("       MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"  RMSE : ${rmse:.2f}")
print(f"  MAE  : ${mae:.2f}")
print(f"  MAPE : {mape:.2f}%")
print("=" * 40)
print(f"\n  Interpretation:")
print(f"  On average, predictions are off by ~${mae:.2f} per share.")
print(f"  That is a {mape:.2f}% error relative to the actual price.")

## Step 10 — Visualise Predictions vs Actual Prices

In [ ]:
# Get corresponding test dates for the x-axis
test_dates = df.index[train_size:]

plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_actual, label='Actual Price',    color='steelblue',  linewidth=1.4)
plt.plot(test_dates, y_pred,   label='Predicted Price', color='darkorange', linewidth=1.4, linestyle='--')
plt.title(f'{TICKER} Stock Price — Actual vs RNN Prediction', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 11 — Predict the Next Day's Price
Use the **most recent 60 trading days** from the downloaded data to predict the very next closing price.

In [ ]:
# Take the last WINDOW_SIZE days of the full dataset
last_window = scaled_data[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)

# Predict
next_scaled = model.predict(last_window, verbose=0)
next_price  = scaler.inverse_transform(next_scaled)[0][0]

last_price = df['Close'].iloc[-1]

print("=" * 40)
print("      NEXT DAY PRICE PREDICTION")
print("=" * 40)
print(f"  Last known closing price : ${last_price:.2f}")
print(f"  Predicted next close     : ${next_price:.2f}")
direction = "UP ↑" if next_price > last_price else "DOWN ↓"
print(f"  Expected direction       : {direction}")
print("=" * 40)
print("\n  Note: This is a learning exercise.")
print("  Do NOT use this for real investment decisions.")

---
## Summary

| Component | Detail |
|---|---|
| Dataset | AAPL closing prices via Yahoo Finance |
| Sequence length | 60 trading days look-back |
| Model | Stacked SimpleRNN (64 → 32 → 1) |
| Activation | tanh (standard for RNN) |
| Optimiser | Adam |
| Loss | Mean Squared Error |
| Metrics | RMSE, MAE, MAPE |

## Key Concepts Covered (Unit 3)
- **RNN:** Processes sequences by carrying hidden state `hₜ` across time steps
- **Vanishing gradient:** Reason why deeper RNNs struggle on very long sequences (→ LSTM solves this)
- **Sequence modelling:** Treating stock prices as a time series
- **Return sequences:** How stacked RNN layers communicate

## Try These Extensions
1. Replace `SimpleRNN` with `LSTM` — compare RMSE
2. Add more features: Volume, Open, High, Low (multivariate RNN)
3. Change the ticker to an Indian stock (e.g., `RELIANCE.NS`)
4. Increase `WINDOW_SIZE` to 90 or 120 days

---
*CSEN3083 Deep Learning | Unit 3 Case Study*